## Fine-Tuning Results Summary

### Comparison with Ed's Frontier Fine-Tuned Model

I trained a fine-tuned **GPT-4.1 Nano** model and compared its performance against Ed’s previously fine-tuned frontier model.

#### Ed's Fine-Tuned Model

* **Mean Error:** $75.09 ± $15.32
* **MSE:** 17,853
* **R²:** 18.8%

#### My Fine-Tuned Model (Epoch = 3)

* **Mean Error:** $57.73 ± $11.37
* **MSE:** 10,062
* **R²:** 54.2%

### Key Improvements

My model shows substantial improvements across all evaluation metrics:

| Metric     | Ed's Model | My Model | Improvement |
| ---------- | ---------- | -------- | ----------- |
| Mean Error | $75.09     | $57.73   | ↓ $17.36    |
| MSE        | 17,853     | 10,062   | ↓ 7,791     |
| R²         | 18.8%      | 54.2%    | +35.4 pts   |

### Conclusion

Using **3 training epochs**, my fine-tuned GPT-4.1 Nano model significantly outperformed Ed’s frontier fine-tuned model. The model achieved:

* **Lower prediction error**
* **Substantially lower MSE**
* **Nearly 3× improvement in R²**

These results indicate that the model captures the underlying pricing patterns much more effectively after fine-tuning.


Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [187]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [188]:
# environment

LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"


train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [186]:
client = OpenAI()

# Data size

OpenAI recommends fine-tuning with a small population of 50-100 examples

I'm going to go with 20,000 points.

This cost me $3.42 - you should stick with 100 examples and the cost will be minimal!

In [189]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


fine_tune_train = train[101:201]
fine_tune_validation = val[51:101]

In [190]:
print(len(fine_tune_train))
print(len(fine_tune_validation))

100
50


# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [191]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [192]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: NSLUMO 2010‑2013 Chevy Camaro LED Fog Light Kit with DRL Halo  \nCategory: Automotive Lighting  \nBrand: NSLUMO  \nDescription: Dual‑function LED fog light and daytime running light kit for 2010‑2013 Chevy Camaro.  \nDetails: 10W LED fog chips and 9.5W LED DRL chips, IP67 waterproof, plug‑and‑play installation with OEM‑sized housing.'},
 {'role': 'assistant', 'content': '$135.99'}]

In [193]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [194]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [195]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [196]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train_7.jsonl")

In [197]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation_7.jsonl")

In [198]:
with open("jsonl/fine_tune_train_7.jsonl", "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")

In [199]:
train_file

FileObject(id='file-V1tdDvi6GZSXCaTvP6g968', bytes=54942, created_at=1772696935, filename='fine_tune_train_7.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [200]:
with open("jsonl/fine_tune_validation_7.jsonl", "rb") as f:
    validation_file = client.files.create(file=f, purpose="fine-tune")

In [201]:
validation_file

FileObject(id='file-NwYsm4H7xCSipGGfZ4s4WG', bytes=28400, created_at=1772696937, filename='fine_tune_validation_7.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [202]:
client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    suffix="pricer_7",
    method = {
        "type": "supervised",
        "supervised": {
            "hyperparameters": {"n_epochs":3}
        }
    }
)

FineTuningJob(id='ftjob-R40mQZJ12PZMPinr0JSJOfu0', created_at=1772696939, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=3), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-J0eG5uJHqRVoMNPBiJ27aTgR', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-V1tdDvi6GZSXCaTvP6g968', validation_file='file-NwYsm4H7xCSipGGfZ4s4WG', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=3))), user_provided_suffix='pricer_7', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [203]:
client.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-R40mQZJ12PZMPinr0JSJOfu0', created_at=1772696939, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=3), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-J0eG5uJHqRVoMNPBiJ27aTgR', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-V1tdDvi6GZSXCaTvP6g968', validation_file='file-NwYsm4H7xCSipGGfZ4s4WG', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=3))), user_provided_suffix='pricer_7', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experimen

In [204]:
job_id = client.fine_tuning.jobs.list(limit=1).data[0].id

In [205]:
job_id

'ftjob-R40mQZJ12PZMPinr0JSJOfu0'

In [206]:
client.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-R40mQZJ12PZMPinr0JSJOfu0', created_at=1772696939, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=3), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-J0eG5uJHqRVoMNPBiJ27aTgR', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-V1tdDvi6GZSXCaTvP6g968', validation_file='file-NwYsm4H7xCSipGGfZ4s4WG', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=3))), user_provided_suffix='pricer_7', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [207]:
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-W3gRb2hfNnjOkxQ0KpEeeO7N', created_at=1772696939, level='info', message='Validating training file: file-V1tdDvi6GZSXCaTvP6g968 and validation file: file-NwYsm4H7xCSipGGfZ4s4WG', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-LqfToZHAFu9FRoYRvuUGD3c1', created_at=1772696939, level='info', message='Created fine-tuning job: ftjob-R40mQZJ12PZMPinr0JSJOfu0', object='fine_tuning.job.event', data={}, type='message')]

https://platform.openai.com/finetune


# Step 3

Test our fine tuned model

In [208]:
fine_tuned_model_name = client.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [217]:
fine_tuned_model_name

In [210]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [211]:
# Try this out

test_messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [ ]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [223]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

219.0
$219.00


In [ ]:
evaluate(gpt_4__1_nano_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $46 $15 $26 $20 $91 $39 $32 $1 $360 $516 $60 $27 $15 $11 $2 $51 $5 $17 $222 $34 $4 $48 $35 $163 $274 $201 $8 $108 $65 $14 $24 $60 $28 $43 $220 $100 $54 $64 $9 $178 $25 $13 $3 $92 $2 $12 $4 $70 $8 $20 $100 $23 $10 $71 $41 $18 $90 $16 $11 $106 $2 $55 $17 $165 $29 $121 $279 $15 $96 $15 $27 $116 $9 $42 $25 $242 $2 $6 $6 $0 $14 $41 $74 $5 $0 $12 $151 $10 $18 $40 $0 $5 $20 $4 $9 $12 $35 $84 $414 $6 $69 $11 $59 $122 $62 $21 $281 $39 $0 $10 $31 $65 $44 $24 $120 $13 $7 $34 $22 $13 $542 $117 $26 $30 $83 $33 $221 $11 $79 $295 $63 $55 $0 $97 $2 $81 $30 $12 $52 $54 $49 $10 $8 $59 $149 $21 $22 $21 $8 $5 $104 $3 $120 $7 $58 $99 $30 $53 $21 $80 $11 $55 $4 $40 $11 $152 $24 $17 $13 $18 $16 $74 $7 $21 $50 $5 $21 $56 $9 $55 $25 $130 $82 $123 $17 $46 $7 $10 $7 $35 $99 $55 $57 $10 $19 $19 $1 $27 $4 